In [1]:
import pyarrow.dataset as ds
print(ds.__doc__)

Dataset is currently unstable. APIs subject to change without notice.


In [2]:
import json
import pandas as pd
from datasets import Dataset
from transformers import MT5Tokenizer

DATA_PATH = "../data/processed/dataset.json"
MODEL_NAME = "google/mt5-small"

c:\Users\win11\anaconda3\envs\nlp-haoussa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Taille dataset:", len(data))
data[:3]

Taille dataset: 1329


[{'translation': {'hausa': 'dalibai', 'zarma': 'lacolize'}},
 {'translation': {'hausa': 'duka !', 'zarma': 'kulu'}},
 {'translation': {'hausa': 'sai gobe ?', 'zarma': 'kalsoubah'}}]

In [4]:
rows = []

for item in data:
    rows.append({
        "hausa": item["translation"]["hausa"],
        "zarma": item["translation"]["zarma"]
    })

df = pd.DataFrame(rows)

df.head()

,hausa,zarma
0,dalibai,lacolize
1,duka !,kulu
2,sai gobe ?,kalsoubah
3,ina kasuwa ?,mano habu?
4,kakata (mace) !,ay kayi


In [5]:
df["hausa"] = df["hausa"].str.strip()
df["zarma"] = df["zarma"].str.strip()

# enlever lignes vides
df = df[(df["hausa"] != "") & (df["zarma"] != "")]

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1063
Validation: 133
Test: 133


In [7]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [8]:
tokenizer = MT5Tokenizer.from_pretrained(MODEL_NAME)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [10]:
PREFIX = "translate Hausa to Zarma: "
MAX_INPUT_LENGTH = 32
MAX_TARGET_LENGTH = 32

def preprocess_function(examples):
    inputs = [PREFIX + text for text in examples["hausa"]]
    targets = examples["zarma"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs
MAX_TARGET_LENGTH = 32

def preprocess_function(examples):
    inputs = [PREFIX + text for text in examples["hausa"]]
    targets = examples["zarma"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [11]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 133/133 [00:00<00:00, 2504.74 examples/s]


In [12]:
tokenized_train[0]

{'hausa': 'Bakwai',
 'zarma': 'iyye',
 'input_ids': [37194,
  10264,
  262,
  288,
  736,
  35716,
  267,
  16900,
  67826,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'labels': [259,
  202857,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [14]:
sample = tokenized_train[0]

decoded_input = tokenizer.decode(sample["input_ids"], skip_special_tokens=True)
decoded_label = tokenizer.decode(sample["labels"], skip_special_tokens=True)

print("INPUT :", decoded_input)
print("TARGET:", decoded_label)

INPUT : translate Hausa to Zarma: Bakwai
TARGET: iyye


In [15]:
tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [16]:
tokenized_train.save_to_disk("../data/processed/train_dataset")
tokenized_val.save_to_disk("../data/processed/val_dataset")
tokenized_test.save_to_disk("../data/processed/test_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 133/133 [00:00<00:00, 7755.78 examples/s]


"""
Observations:

1. Données converties en format compatible Transformer
2. Utilisation du préfixe pour guider mT5
3. Longueur max fixée à 32 (adaptée aux phrases courtes)
4. Padding appliqué pour batch training
5. Labels correctement alignés avec targets

Conclusion:
Dataset prêt pour fine-tuning.
"""